In [ ]:
# Set up Step 5.
#
# Step 4 outputs are read from cfg.STEP5_INPUT_DIR.
# This run uses the model specified by cfg.STEP5_MODEL_NAME.

import os
import sys
import shutil
import importlib
from google.colab import files, drive

drive.mount("/content/drive")

PILOT_ROOT = "/content/pilot_full"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

print("Upload config.py / config_positional_context_full_drive.py")
uploaded = files.upload()

config_names = [
    name for name in uploaded.keys()
    if name.startswith("config") and name.endswith(".py")
]

if len(config_names) != 1:
    raise ValueError(
        f"Expected exactly one config*.py file, got: {config_names}"
    )

config_src = os.path.join("/content", config_names[0])
config_dst = os.path.join(PILOT_ROOT, "config.py")
shutil.copy(config_src, config_dst)

if "pilot_full.config" in sys.modules:
    del sys.modules["pilot_full.config"]

import pilot_full.config as cfg
importlib.reload(cfg)

print("Loaded config:", config_names[0])
print("PIPELINE_DATA_DIR:", cfg.PIPELINE_DATA_DIR)
print("STEP5_INPUT_DIR:", cfg.STEP5_INPUT_DIR)
print("STEP5_MODEL_NAME:", cfg.STEP5_MODEL_NAME)
print("STEP5_MODEL_TAG:", cfg.STEP5_MODEL_TAG)
print("STEP5_SAMPLE_FAMILIES:", cfg.STEP5_SAMPLE_FAMILIES)

for sample_family in cfg.STEP5_SAMPLE_FAMILIES:
    out_dir = cfg.make_step5_output_dir(sample_family)
    os.makedirs(out_dir, exist_ok=True)
    print("Output dir:", sample_family, "->", out_dir)

Mounted at /content/drive
Upload config.py / config_positional_context_full_drive.py


Saving config_positional_context_full_drive.py to config_positional_context_full_drive.py
Loaded config: config_positional_context_full_drive.py
PIPELINE_DATA_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data
STEP5_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe
STEP5_MODEL_NAME: Qwen/Qwen2.5-0.5B
STEP5_MODEL_TAG: qwen2_5_0_5b
STEP5_SAMPLE_FAMILIES: ['pair', 'composition']
Output dir: pair -> /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_0_5b
Output dir: composition -> /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_composition_qwen2_5_0_5b


In [ ]:
import os
import sys
import json
import random
import shutil
import importlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import torch
from tqdm.auto import tqdm

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_full.config as cfg
importlib.reload(cfg)

MODEL_NAME = cfg.STEP5_MODEL_NAME
MODEL_TAG = cfg.STEP5_MODEL_TAG

MAX_LENGTH = cfg.STEP5_MAX_LENGTH
RANDOM_SEED = cfg.RANDOM_SEED
RUN_MODE = cfg.RUN_MODE

STEP5_INPUT_DIR = cfg.STEP5_INPUT_DIR
STEP5_SAMPLE_FAMILIES = cfg.STEP5_SAMPLE_FAMILIES
STEP5_USE_PER_SCENE_FILES = cfg.STEP5_USE_PER_SCENE_FILES

STEP5_PAIR_INPUT_GLOB = cfg.STEP5_PAIR_INPUT_GLOB
STEP5_COMPOSITION_INPUT_GLOB = cfg.STEP5_COMPOSITION_INPUT_GLOB

STEP5_PAIR_ALL_INPUT_FILE = cfg.STEP5_PAIR_ALL_INPUT_FILE
STEP5_COMPOSITION_ALL_INPUT_FILE = cfg.STEP5_COMPOSITION_ALL_INPUT_FILE

SOURCE_TYPE = cfg.STEP5_SOURCE_TYPE
INPUT_TEXT_MODE = cfg.STEP5_INPUT_TEXT_MODE
FEATURE_ANCHOR = cfg.STEP5_FEATURE_ANCHOR
FORWARD_TEXT_ONLY = cfg.STEP5_FORWARD_TEXT_ONLY
USE_ALL_ALIAS_OCCURRENCES = cfg.STEP5_USE_ALL_ALIAS_OCCURRENCES
FEATURE_DTYPE = cfg.STEP5_FEATURE_DTYPE

PRINT_FILE_SUMMARY = True
PRINT_SKIP_EXAMPLES = True
MAX_SKIP_EXAMPLES = 10

random.seed(RANDOM_SEED)

print("Step 5 config loaded.")
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("MAX_LENGTH:", MAX_LENGTH)
print("RUN_MODE:", RUN_MODE)
print("STEP5_INPUT_DIR:", STEP5_INPUT_DIR)
print("STEP5_SAMPLE_FAMILIES:", STEP5_SAMPLE_FAMILIES)
print("STEP5_USE_PER_SCENE_FILES:", STEP5_USE_PER_SCENE_FILES)

Step 5 config loaded.
MODEL_NAME: Qwen/Qwen2.5-0.5B
MODEL_TAG: qwen2_5_0_5b
MAX_LENGTH: 768
RUN_MODE: diverse
STEP5_INPUT_DIR: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step4_probe
STEP5_SAMPLE_FAMILIES: ['pair', 'composition']
STEP5_USE_PER_SCENE_FILES: True


In [ ]:
# Load tokenizer and model

from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        dtype = torch.bfloat16
        device_map = "auto"
    else:
        dtype = torch.float32
        device_map = None

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map=device_map,
    )

    model.eval()

    return tokenizer, model


print("Model loader defined.")

Model loader defined.


In [ ]:
def print_device_and_model_info(model):
    print("CUDA environment check:")
    print("torch.cuda.is_available():", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("CUDA device count:", torch.cuda.device_count())
        print("CUDA device name:", torch.cuda.get_device_name(0))
        print("CUDA capability:", torch.cuda.get_device_capability(0))
        print("CUDA allocated memory GB:", round(torch.cuda.memory_allocated(0) / 1024**3, 3))
        print("CUDA reserved memory GB:", round(torch.cuda.memory_reserved(0) / 1024**3, 3))
    else:
        print("CUDA is not available. The runtime is using CPU.")

    first_param = next(model.parameters())
    print("Model dtype:", first_param.dtype)
    print("Model device:", first_param.device)

In [ ]:
# Utility functions

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows


def write_json(path: str, data: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def discover_step5_input_files(sample_family: str) -> List[str]:
    """
    Discover Step4 input files for one sample family.

    Per-scene mode:
        FloorPlan*_step4_probe_samples_{RUN_MODE}.jsonl
        FloorPlan*_step4_composition_samples_{RUN_MODE}.jsonl

    Aggregate mode:
        step4_probe_samples_{RUN_MODE}_all.jsonl
        step4_composition_samples_{RUN_MODE}_all.jsonl
    """
    if sample_family == "pair":
        if STEP5_USE_PER_SCENE_FILES:
            paths = sorted(Path(STEP5_INPUT_DIR).glob(STEP5_PAIR_INPUT_GLOB))
        else:
            paths = [Path(STEP5_PAIR_ALL_INPUT_FILE)]

    elif sample_family == "composition":
        if STEP5_USE_PER_SCENE_FILES:
            paths = sorted(Path(STEP5_INPUT_DIR).glob(STEP5_COMPOSITION_INPUT_GLOB))
        else:
            paths = [Path(STEP5_COMPOSITION_ALL_INPUT_FILE)]

    else:
        raise ValueError(f"Unsupported sample_family: {sample_family}")

    paths = [str(p) for p in paths if p.exists()]

    if not paths:
        raise FileNotFoundError(
            f"No Step4 input files found for sample_family={sample_family} "
            f"in {STEP5_INPUT_DIR}"
        )

    return paths


def normalize_char_span(span) -> Optional[Dict[str, int]]:
    """
    Accept either:
        {"start": int, "end": int}
    or:
        [start, end]
    or:
        (start, end)
    """
    if span is None:
        return None

    if isinstance(span, dict):
        start = span.get("start")
        end = span.get("end")
    elif isinstance(span, (list, tuple)) and len(span) == 2:
        start, end = span
    else:
        return None

    if start is None or end is None:
        return None

    start = int(start)
    end = int(end)

    if end <= start:
        return None

    return {"start": start, "end": end}


def normalize_char_spans(spans) -> List[Dict[str, int]]:
    if not spans:
        return []

    out = []

    for span in spans:
        norm = normalize_char_span(span)
        if norm is not None:
            out.append(norm)

    return out


def char_span_to_token_span(
    offsets: List[Tuple[int, int]],
    char_span: Dict[str, int],
) -> Optional[Tuple[int, int]]:
    char_start = char_span["start"]
    char_end = char_span["end"]

    token_indices = []

    for tok_idx, offset in enumerate(offsets):
        tok_start, tok_end = offset

        if tok_start == tok_end:
            continue

        overlap = min(tok_end, char_end) - max(tok_start, char_start)

        if overlap > 0:
            token_indices.append(tok_idx)

    if not token_indices:
        return None

    return (token_indices[0], token_indices[-1] + 1)


def mean_multiple_token_spans(
    token_hidden: torch.Tensor,
    token_spans: List[Tuple[int, int]],
) -> torch.Tensor:
    vectors = []

    for start, end in token_spans:
        if start < 0 or end <= start or end > token_hidden.shape[0]:
            continue

        vectors.append(token_hidden[start:end].mean(dim=0))

    if not vectors:
        raise ValueError("No valid token spans found.")

    return torch.stack(vectors, dim=0).mean(dim=0)


def validate_current_step4_sample(row: Dict[str, Any]) -> None:
    required = {
        "sample_id",
        "sample_type",
        "pair_group_id",
        "scene",
        "paragraph_id",
        "text",
        "subject_uid",
        "object_uid",
        "relation",
        "evidence_type",
        "subject_all_char_spans",
        "object_all_char_spans",
    }

    missing = [k for k in required if k not in row]

    if missing:
        raise KeyError(
            f"Step4 row missing required fields {missing}; "
            f"sample_id={row.get('sample_id')}"
        )


def feature_to_dtype(t: torch.Tensor) -> torch.Tensor:
    if FEATURE_DTYPE == "float16":
        return t.to(torch.float16)

    if FEATURE_DTYPE == "float32":
        return t.to(torch.float32)

    raise ValueError(f"Unsupported FEATURE_DTYPE: {FEATURE_DTYPE}")


print("Utility functions loaded.")

Utility functions loaded.


In [ ]:
# Build Step5 examples from current Step4 samples

def build_step5_example_from_step4_sample(row: Dict[str, Any]) -> Dict[str, Any]:
    validate_current_step4_sample(row)

    text = row["text"]

    subject_uid = row["subject_uid"]
    object_uid = row["object_uid"]

    subject_all_char_spans = normalize_char_spans(
        row.get("subject_all_char_spans")
    )
    object_all_char_spans = normalize_char_spans(
        row.get("object_all_char_spans")
    )

    return {
        # Identity
        "sample_id": row["sample_id"],
        "sample_type": row.get("sample_type"),
        "composition_type": row.get("composition_type"),
        "pair_group_id": row["pair_group_id"],

        "scene": row.get("scene"),
        "scene_type": row.get("scene_type"),
        "paragraph_id": row.get("paragraph_id"),
        "chunk_id": row.get("chunk_id"),
        "cluster_id": row.get("cluster_id"),
        "paragraph_index_within_cluster": row.get("paragraph_index_within_cluster"),
        "generation_mode": row.get("generation_mode"),

        # Only this field is fed into LLM
        "text": text,

        # Label
        "relation": row["relation"],
        "relation_label": row.get("relation_label", row["relation"]),
        "has_relation_label": row.get("has_relation_label"),
        "label_is_none": row.get("label_is_none"),

        # Probe pair
        "subject_alias": subject_uid,
        "object_alias": object_uid,
        "subject_uid": subject_uid,
        "object_uid": object_uid,
        "subject_id": row.get("subject_id"),
        "object_id": row.get("object_id"),
        "subject_type": row.get("subject_type"),
        "object_type": row.get("object_type"),

        # Character spans
        "subject_all_char_spans": subject_all_char_spans,
        "object_all_char_spans": object_all_char_spans,

        # Evidence metadata
        "evidence_type": row.get("evidence_type"),
        "pair_evidence_type": row.get("pair_evidence_type"),
        "target_pair_evidence_type": row.get("target_pair_evidence_type"),
        "probe_direction_relative_to_text": row.get("probe_direction_relative_to_text"),
        "is_inverse_of_text_relation": row.get("is_inverse_of_text_relation"),
        "relation_source": row.get("relation_source"),

        # Metadata copied for later Step6 / analysis, not fed to LLM
        "geometry_label": row.get("geometry_label"),
        "source_relation_summary": row.get("source_relation_summary"),

        # Composition metadata
        "target_subject_uid": row.get("target_subject_uid"),
        "target_object_uid": row.get("target_object_uid"),
        "target_relation": row.get("target_relation"),
        "num_supporting_paths": row.get("num_supporting_paths"),
        "supporting_paths": row.get("supporting_paths"),
        "primary_support": row.get("primary_support"),
        "bridge_uids": row.get("bridge_uids"),
        "implicit_rules": row.get("implicit_rules"),

        # Keep original row for debugging if needed
        "source_row": row,
    }


def build_step5_examples(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [
        build_step5_example_from_step4_sample(row)
        for row in rows
    ]


print("Step5 example builder loaded.")

Step5 example builder loaded.


In [ ]:
# Preflight check: token length and alias span coverage

def check_token_length_and_span_coverage(
    input_files: List[str],
    tokenizer,
    max_length: int = MAX_LENGTH,
    max_bad_examples: int = 10,
) -> Dict[str, Any]:
    total_examples = 0
    token_lengths_no_trunc = []
    truncated_count = 0
    span_failed_count = 0
    bad_examples = []

    for input_path in input_files:
        rows = load_jsonl(input_path)
        examples = build_step5_examples(rows)

        for ex in examples:
            total_examples += 1
            text = ex["text"]

            enc_full = tokenizer(
                text,
                return_tensors="pt",
                truncation=False,
                add_special_tokens=True,
            )

            true_len = int(enc_full["input_ids"].shape[1])
            token_lengths_no_trunc.append(true_len)

            if true_len > max_length:
                truncated_count += 1

            enc = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                return_offsets_mapping=True,
                add_special_tokens=True,
            )

            offsets = enc["offset_mapping"][0].tolist()

            subject_token_spans = [
                char_span_to_token_span(offsets, span)
                for span in ex["subject_all_char_spans"]
            ]
            subject_token_spans = [
                span for span in subject_token_spans
                if span is not None
            ]

            object_token_spans = [
                char_span_to_token_span(offsets, span)
                for span in ex["object_all_char_spans"]
            ]
            object_token_spans = [
                span for span in object_token_spans
                if span is not None
            ]

            if not subject_token_spans or not object_token_spans:
                span_failed_count += 1

                if len(bad_examples) < max_bad_examples:
                    bad_examples.append({
                        "sample_id": ex["sample_id"],
                        "sample_type": ex.get("sample_type"),
                        "source_file": os.path.basename(input_path),
                        "true_num_tokens": true_len,
                        "subject_alias": ex["subject_alias"],
                        "object_alias": ex["object_alias"],
                        "num_subject_char_spans": len(ex["subject_all_char_spans"]),
                        "num_object_char_spans": len(ex["object_all_char_spans"]),
                    })

    return {
        "total_examples": total_examples,
        "max_length": max_length,
        "max_tokens_no_trunc": max(token_lengths_no_trunc) if token_lengths_no_trunc else 0,
        "mean_tokens_no_trunc": sum(token_lengths_no_trunc) / len(token_lengths_no_trunc) if token_lengths_no_trunc else 0,
        "truncated_count": truncated_count,
        "span_failed_count": span_failed_count,
        "bad_examples": bad_examples,
    }


print("Preflight function loaded.")

Preflight function loaded.


In [ ]:
# Hidden-state extraction for one paragraph group

@torch.no_grad()
def forward_paragraph_text(
    text: str,
    tokenizer,
    model,
    max_length: int,
) -> Dict[str, Any]:
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    offsets = enc["offset_mapping"][0].tolist()

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    model_device = next(model.parameters()).device

    outputs = model(
        input_ids=input_ids.to(model_device),
        attention_mask=attention_mask.to(model_device),
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )

    hidden_states = [
        h.squeeze(0).detach().cpu()
        for h in outputs.hidden_states
    ]

    decoded_tokens = tokenizer.convert_ids_to_tokens(
        enc["input_ids"][0].tolist()
    )

    return {
        "input_ids": enc["input_ids"][0].cpu(),
        "attention_mask": enc["attention_mask"][0].cpu(),
        "offset_mapping": offsets,
        "decoded_tokens": decoded_tokens,
        "hidden_states": hidden_states,
    }


def extract_record_from_paragraph_forward(
    ex: Dict[str, Any],
    paragraph_forward: Dict[str, Any],
) -> Tuple[Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    offsets = paragraph_forward["offset_mapping"]

    subject_token_spans = [
        char_span_to_token_span(offsets, span)
        for span in ex["subject_all_char_spans"]
    ]
    subject_token_spans = [
        span for span in subject_token_spans
        if span is not None
    ]

    object_token_spans = [
        char_span_to_token_span(offsets, span)
        for span in ex["object_all_char_spans"]
    ]
    object_token_spans = [
        span for span in object_token_spans
        if span is not None
    ]

    if not subject_token_spans or not object_token_spans:
        return None, {
            "sample_id": ex.get("sample_id"),
            "sample_type": ex.get("sample_type"),
            "reason": "no_visible_subject_or_object_token_spans",
            "subject_alias": ex.get("subject_alias"),
            "object_alias": ex.get("object_alias"),
            "num_subject_char_spans": len(ex.get("subject_all_char_spans") or []),
            "num_object_char_spans": len(ex.get("object_all_char_spans") or []),
        }

    layer_subject_vectors = []
    layer_object_vectors = []
    layer_diff_features = []
    layer_concat_features = []

    for token_h in paragraph_forward["hidden_states"]:
        h_subject = mean_multiple_token_spans(
            token_h,
            subject_token_spans,
        )

        h_object = mean_multiple_token_spans(
            token_h,
            object_token_spans,
        )

        diff = h_subject - h_object
        concat = torch.cat([h_subject, h_object], dim=0)

        layer_subject_vectors.append(h_subject)
        layer_object_vectors.append(h_object)
        layer_diff_features.append(diff)
        layer_concat_features.append(concat)

    record = {
        # Identity
        "sample_id": ex["sample_id"],
        "sample_type": ex.get("sample_type"),
        "composition_type": ex.get("composition_type"),
        "pair_group_id": ex["pair_group_id"],

        "scene": ex.get("scene"),
        "scene_type": ex.get("scene_type"),
        "paragraph_id": ex.get("paragraph_id"),
        "chunk_id": ex.get("chunk_id"),
        "cluster_id": ex.get("cluster_id"),
        "paragraph_index_within_cluster": ex.get("paragraph_index_within_cluster"),
        "generation_mode": ex.get("generation_mode"),

        # LLM input text
        "text": ex["text"],

        # Label
        "relation": ex["relation"],
        "relation_label": ex.get("relation_label", ex["relation"]),
        "has_relation_label": ex.get("has_relation_label"),
        "label_is_none": ex.get("label_is_none"),

        # Probe pair
        "subject_alias": ex["subject_alias"],
        "object_alias": ex["object_alias"],
        "subject_uid": ex.get("subject_uid"),
        "object_uid": ex.get("object_uid"),
        "subject_id": ex.get("subject_id"),
        "object_id": ex.get("object_id"),
        "subject_type": ex.get("subject_type"),
        "object_type": ex.get("object_type"),

        # Evidence
        "evidence_type": ex.get("evidence_type"),
        "pair_evidence_type": ex.get("pair_evidence_type"),
        "target_pair_evidence_type": ex.get("target_pair_evidence_type"),
        "probe_direction_relative_to_text": ex.get("probe_direction_relative_to_text"),
        "is_inverse_of_text_relation": ex.get("is_inverse_of_text_relation"),
        "relation_source": ex.get("relation_source"),

        # Metadata, not fed to LLM
        "geometry_label": ex.get("geometry_label"),
        "source_relation_summary": ex.get("source_relation_summary"),

        # Composition metadata
        "target_subject_uid": ex.get("target_subject_uid"),
        "target_object_uid": ex.get("target_object_uid"),
        "target_relation": ex.get("target_relation"),
        "num_supporting_paths": ex.get("num_supporting_paths"),
        "supporting_paths": ex.get("supporting_paths"),
        "primary_support": ex.get("primary_support"),
        "bridge_uids": ex.get("bridge_uids"),
        "implicit_rules": ex.get("implicit_rules"),

        # Tokenization metadata
        "input_ids": paragraph_forward["input_ids"],
        "attention_mask": paragraph_forward["attention_mask"],
        "offset_mapping": paragraph_forward["offset_mapping"],
        "decoded_tokens": paragraph_forward["decoded_tokens"],
        "num_tokens": len(paragraph_forward["decoded_tokens"]),
        "max_length": MAX_LENGTH,

        # Span metadata
        "subject_all_char_spans": ex.get("subject_all_char_spans"),
        "object_all_char_spans": ex.get("object_all_char_spans"),
        "subject_token_spans": subject_token_spans,
        "object_token_spans": object_token_spans,

        # Hidden-state features
        "layer_subject_vectors": feature_to_dtype(torch.stack(layer_subject_vectors, dim=0)),
        "layer_object_vectors": feature_to_dtype(torch.stack(layer_object_vectors, dim=0)),
        "layer_diff_features": feature_to_dtype(torch.stack(layer_diff_features, dim=0)),
        "layer_concat_features": feature_to_dtype(torch.stack(layer_concat_features, dim=0)),
    }

    return record, None


print("Paragraph-level hidden-state extraction functions loaded.")

Paragraph-level hidden-state extraction functions loaded.


In [ ]:
# Process one Step4 file

def process_step4_file(
    input_path: str,
    sample_family: str,
    model_name: str,
    model_tag: str,
    tokenizer,
    model,
    output_dir: str,
) -> Dict[str, Any]:
    rows = load_jsonl(input_path)
    examples = build_step5_examples(rows)

    source_file = os.path.basename(input_path)

    examples_by_paragraph = defaultdict(list)
    for ex in examples:
        paragraph_key = (
            ex.get("scene"),
            ex.get("paragraph_id"),
            ex["text"],
        )
        examples_by_paragraph[paragraph_key].append(ex)

    scenes = sorted({
        ex.get("scene")
        for ex in examples
        if ex.get("scene") is not None
    })
    scene = scenes[0] if len(scenes) == 1 else "mixed_scenes"

    kept_records = []
    skipped_records = []

    for paragraph_key, paragraph_examples in tqdm(
        examples_by_paragraph.items(),
        desc=f"Forward paragraphs: {source_file}",
    ):
        text = paragraph_examples[0]["text"]

        try:
            paragraph_forward = forward_paragraph_text(
                text=text,
                tokenizer=tokenizer,
                model=model,
                max_length=MAX_LENGTH,
            )
        except Exception as e:
            for ex in paragraph_examples:
                skipped_records.append({
                    "sample_id": ex.get("sample_id"),
                    "sample_type": ex.get("sample_type"),
                    "reason": "paragraph_forward_failed",
                    "error": repr(e),
                })
            continue

        for ex in paragraph_examples:
            record, skip_info = extract_record_from_paragraph_forward(
                ex,
                paragraph_forward,
            )

            if record is not None:
                kept_records.append(record)
            else:
                skipped_records.append(skip_info)

    if kept_records:
        num_layers = kept_records[0]["layer_diff_features"].shape[0]
        feature_dim = kept_records[0]["layer_diff_features"].shape[1]
        concat_dim = kept_records[0]["layer_concat_features"].shape[1]
    else:
        num_layers = None
        feature_dim = None
        concat_dim = None

    output_filename = cfg.make_step5_output_pt_filename(
        source_filename=source_file,
        sample_family=sample_family,
    )

    output_pt_path = os.path.join(
        output_dir,
        output_filename,
    )

    summary = {
        "source_file": source_file,
        "sample_family": sample_family,
        "scene": scene,

        "source_type": SOURCE_TYPE,
        "model_name": model_name,
        "model_tag": model_tag,
        "input_text_mode": INPUT_TEXT_MODE,
        "feature_anchor": FEATURE_ANCHOR,
        "forward_text_only": FORWARD_TEXT_ONLY,

        "run_mode": RUN_MODE,
        "max_length": MAX_LENGTH,

        "num_input_examples": len(examples),
        "num_unique_paragraphs": len(examples_by_paragraph),
        "num_kept_examples": len(kept_records),
        "num_skipped_examples": len(skipped_records),

        "num_layers": num_layers,
        "feature_dim": feature_dim,
        "concat_dim": concat_dim,

        "label_counts": dict(Counter(r["relation"] for r in kept_records)),
        "sample_type_counts": dict(Counter(r["sample_type"] for r in kept_records)),
        "evidence_type_counts": dict(Counter(r["evidence_type"] for r in kept_records)),
        "probe_direction_counts": dict(
            Counter(r.get("probe_direction_relative_to_text") for r in kept_records)
        ),
        "skip_reason_counts": dict(Counter(s["reason"] for s in skipped_records)),
    }

    payload = {
        **summary,
        "records": kept_records,
        "skipped_records": skipped_records,
    }

    torch.save(payload, output_pt_path)

    summary["output_pt_path"] = output_pt_path

    return summary


print("File processing function loaded.")

File processing function loaded.


In [ ]:
# Step5 hidden-state extraction

all_run_summaries = []

print("\n" + "=" * 100)
print("Loading active model:", MODEL_NAME)
print("MODEL_TAG:", MODEL_TAG)
print("=" * 100)

tokenizer, model = load_model_and_tokenizer(MODEL_NAME)
print_device_and_model_info(model)

for sample_family in STEP5_SAMPLE_FAMILIES:
    output_dir = cfg.make_step5_output_dir(sample_family)
    os.makedirs(output_dir, exist_ok=True)

    input_files = discover_step5_input_files(sample_family)

    print("\n" + "-" * 100)
    print("sample_family:", sample_family)
    print("output_dir:", output_dir)
    print("input files:", len(input_files))
    for path in input_files[:10]:
        print("-", os.path.basename(path))
    if len(input_files) > 10:
        print("...")

    file_summaries = []

    for input_path in input_files:
        summary = process_step4_file(
            input_path=input_path,
            sample_family=sample_family,
            model_name=MODEL_NAME,
            model_tag=MODEL_TAG,
            tokenizer=tokenizer,
            model=model,
            output_dir=output_dir,
        )
        file_summaries.append(summary)

        if PRINT_FILE_SUMMARY:
            print(
                f"[Done] {summary['source_file']} | "
                f"family={sample_family} | "
                f"scene={summary['scene']} | "
                f"paragraphs={summary['num_unique_paragraphs']} | "
                f"kept={summary['num_kept_examples']} | "
                f"skipped={summary['num_skipped_examples']}"
            )

    run_summary = {
        "model_name": MODEL_NAME,
        "model_tag": MODEL_TAG,
        "sample_family": sample_family,
        "output_dir": output_dir,
        "num_processed_files": len(file_summaries),
        "file_summaries": file_summaries,
    }

    all_run_summaries.append(run_summary)

    summary_path = os.path.join(
        output_dir,
        cfg.make_step5_run_summary_filename(sample_family),
    )
    write_json(summary_path, run_summary)

del model
del tokenizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nStep5 processing complete.")
print("Active model:", MODEL_NAME)
print("Total sample-family runs:", len(all_run_summaries))


Loading active model: Qwen/Qwen2.5-0.5B
MODEL_TAG: qwen2_5_0_5b


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[Done] FloorPlan7_step4_composition_samples_diverse.jsonl | family=composition | scene=FloorPlan7 | paragraphs=7 | kept=30 | skipped=0


Forward paragraphs: FloorPlan8_step4_composition_samples_diverse.jsonl:   0%|          | 0/5 [00:00<?, ?it/s]

[Done] FloorPlan8_step4_composition_samples_diverse.jsonl | family=composition | scene=FloorPlan8 | paragraphs=5 | kept=12 | skipped=0


Forward paragraphs: FloorPlan9_step4_composition_samples_diverse.jsonl:   0%|          | 0/11 [00:00<?, ?it/s]

[Done] FloorPlan9_step4_composition_samples_diverse.jsonl | family=composition | scene=FloorPlan9 | paragraphs=11 | kept=56 | skipped=0

Step5 processing complete.
Active model: Qwen/Qwen2.5-0.5B
Total sample-family runs: 2


In [ ]:
global_summary_path = cfg.make_step5_global_summary_path()

global_summary = {
    "run_mode": RUN_MODE,
    "model_name": MODEL_NAME,
    "model_tag": MODEL_TAG,
    "sample_families": STEP5_SAMPLE_FAMILIES,
    "max_length": MAX_LENGTH,
    "step5_input_dir": STEP5_INPUT_DIR,
    "num_runs": len(all_run_summaries),
    "runs": all_run_summaries,
}

write_json(global_summary_path, global_summary)

print("Global Step5 summary:", global_summary_path)
print(json.dumps(
    {
        "num_runs": global_summary["num_runs"],
        "model_tag": global_summary["model_tag"],
        "sample_families": global_summary["sample_families"],
    },
    indent=2,
    ensure_ascii=False,
))

Global Step5 summary: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hidden_state_global_summary_qwen2_5_0_5b_diverse.json
{
  "num_runs": 2,
  "model_tag": "qwen2_5_0_5b",
  "sample_families": [
    "pair",
    "composition"
  ]
}


In [ ]:
all_pt_files = []

for sample_family in STEP5_SAMPLE_FAMILIES:
    output_dir = cfg.make_step5_output_dir(sample_family)
    pt_files = sorted(Path(output_dir).glob(f"*_step5_hs_{MODEL_TAG}.pt"))

    print("\nOutput dir:", output_dir)
    print("PT files:", len(pt_files))

    for p in pt_files[:10]:
        print("-", p.name)

    all_pt_files.extend(pt_files)

if not all_pt_files:
    raise FileNotFoundError("No Step5 .pt files found.")

sample_payload = torch.load(all_pt_files[0], map_location="cpu")

print("\nLoaded sample payload:")
print("source_file:", sample_payload.get("source_file"))
print("sample_family:", sample_payload.get("sample_family"))
print("scene:", sample_payload.get("scene"))
print("model_name:", sample_payload.get("model_name"))
print("model_tag:", sample_payload.get("model_tag"))
print("num_kept_examples:", sample_payload.get("num_kept_examples"))
print("num_skipped_examples:", sample_payload.get("num_skipped_examples"))
print("num_layers:", sample_payload.get("num_layers"))
print("feature_dim:", sample_payload.get("feature_dim"))
print("concat_dim:", sample_payload.get("concat_dim"))

records = sample_payload.get("records", [])
print("records:", len(records))

if records:
    r = records[0]
    print("\nFirst record keys:")
    print(sorted(r.keys()))

    print("\nFeature shapes:")
    print("layer_subject_vectors:", tuple(r["layer_subject_vectors"].shape))
    print("layer_object_vectors:", tuple(r["layer_object_vectors"].shape))
    print("layer_diff_features:", tuple(r["layer_diff_features"].shape))
    print("layer_concat_features:", tuple(r["layer_concat_features"].shape))

    print("\nLabel/evidence:")
    print("sample_type:", r.get("sample_type"))
    print("relation:", r.get("relation"))
    print("evidence_type:", r.get("evidence_type"))


Output dir: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_0_5b
PT files: 118
- FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan11_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan12_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan13_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan14_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan15_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan16_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan17_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan18_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
- FloorPlan19_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt

Output dir: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_composition_qwen2_5_0_5b
PT files: 94
- FloorPlan10_step4_composition_samples_diverse_st

In [ ]:
from google.colab import files

for sample_family in STEP5_SAMPLE_FAMILIES:
    output_dir = cfg.make_step5_output_dir(sample_family)

    if not os.path.exists(output_dir):
        print("[Skip missing output dir]", output_dir)
        continue

    output_files = sorted(os.listdir(output_dir))

    if not output_files:
        print("[Skip empty output dir]", output_dir)
        continue

    zip_base = f"/content/pilot_full_step5_hs_{sample_family}_{MODEL_TAG}"

    zip_path = shutil.make_archive(
        base_name=zip_base,
        format="zip",
        root_dir=output_dir,
    )

    print("Created zip:", zip_path)
    print("Files included:", len(output_files))

    files.download(zip_path)

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import os

for sample_family in STEP5_SAMPLE_FAMILIES:
    output_dir = cfg.make_step5_output_dir(sample_family)
    pt_files = sorted(Path(output_dir).glob(f"*_step5_hs_{MODEL_TAG}.pt"))
    json_files = sorted(Path(output_dir).glob("*.json"))

    print("=" * 80)
    print("sample_family:", sample_family)
    print("output_dir:", output_dir)
    print("pt files:", len(pt_files))
    print("json summary files:", len(json_files))

    if pt_files:
        print("first pt:", pt_files[0].name)
        print("last pt:", pt_files[-1].name)
        print("total size GB:", round(sum(p.stat().st_size for p in pt_files) / 1024**3, 2))

sample_family: pair
output_dir: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_pair_qwen2_5_0_5b
pt files: 118
json summary files: 1
first pt: FloorPlan10_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
last pt: FloorPlan9_step4_probe_samples_diverse_step5_hs_qwen2_5_0_5b.pt
total size GB: 32.44
sample_family: composition
output_dir: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/step5_hs_composition_qwen2_5_0_5b
pt files: 94
json summary files: 1
first pt: FloorPlan10_step4_composition_samples_diverse_step5_hs_qwen2_5_0_5b.pt
last pt: FloorPlan9_step4_composition_samples_diverse_step5_hs_qwen2_5_0_5b.pt
total size GB: 0.34
